In [ ]:
import pandas as pd
import pickle
from src.utils import *
from src.embedding_analyzer_codenet import EmbeddingAnalyzerNet
from src.no_para_analzyer import No_Para_Analyzer

device = getting_device()
  
train_df =pd.read_csv('codenet/train_1725993.csv')
test_df = pd.read_csv('codenet/test_1824.csv')
desc_df = pd.read_csv('codenet/test_55.csv')

model_names = ['bert', 'gpt', 'roberta', 'falcon7b', 'falcon11b', 'falcon40b', 'llama7b', 'llama8b', 'llama13b',  'llama70b', 'embedding_ada', 'embedding_small', 'embedding_large']
classifier_names = ['KNN', 'SVM', 'CNN']
goals = ['lang', 'task', 'desc']
r_dic = {model: {classifier: {goal: None for goal in goals} for classifier in classifier_names} for model in model_names}

## KNN

In [ ]:
c = 'KNN'
for j in [5, 10]:
    params = {'n_neighbors': j}
    for m in model_names:
        print('Model: {}; Classifier: {}'.format(m, c))
        analyzer = No_Para_Analyzer(model_name = m , classifier = c, params=params)
        r_dic[m][c][goals[0]], r_dic[m][c][goals[1]], r_dic[m][c][goals[2]] = analyzer.analyze(train_df, test_df, desc_df)
        print('--------------------------------------------------------')


## CNN

In [ ]:
setting_1 = {'l_r': 0.0001, 'b_size': 128, 'stop_threshold': 0.05}
setting_2 = {'l_r': 0.0001, 'b_size': 128, 'stop_threshold': 0.01}
setting_3 = {'l_r': 0.0001, 'b_size': 128, 'stop_threshold': 0.001}
setting_4 = {'l_r': 0.0001, 'b_size': 128, 'stop_threshold': 0.005}
setting_5 = {'l_r': 0.0001, 'b_size': 256, 'stop_threshold': 0.005}
setting_6 = {'l_r': 0.0001, 'b_size': 512, 'stop_threshold': 0.005}
setting_7 = {'l_r': 0.01, 'b_size': 128, 'stop_threshold': 0.005}
setting_8 = {'l_r': 0.001, 'b_size': 128, 'stop_threshold': 0.001}
setting_9 = {'l_r': 0.00001, 'b_size': 128, 'stop_threshold': 0.005}

In [ ]:
c = 'CNN'
for m in model_names:
    if r_dic[m][c][goals[0]] is None:
        print('Model: {}; Classifier: {}'.format(m, c))
        analyzer = No_Para_Analyzer(model_name = m , classifier = c, params =setting_3 )
        r_dic[m][c][goals[0]], r_dic[m][c][goals[1]], r_dic[m][c][goals[2]] = analyzer.analyze(train_df, test_df, desc_df)
    else:
        print('Pass as already caculated!')
        pass

## CNN Log Parser

In [ ]:
for file_name in range(1, 10):
    file_name = str(file_name)
    log_file = "log/transfer/setting"+file_name+".log" 
    results = parse_log_file(log_file)

    df = pd.DataFrame(results)
    df.to_csv("log/transfer/setting"+file_name+".csv", index=False)